# APEX Go2 を MuJoCo Warp で動かす

元 APEX の Go2 平坦地形設定を対象に、物理シミュレータを MuJoCo Warp に差し替えます。学習器は元リポジトリの Multi-Critic PPO を使用します。Linux、NVIDIA CUDA GPU、Python 3.10 以降を想定します。まず少数環境で観測と報酬を確認してから学習してください。


## 1. 依存ライブラリ

CUDA 対応 PyTorch は [公式手順](https://pytorch.org/get-started/locally/)に従って先に導入してください。以下のセルは MuJoCo Warp と学習記録用ライブラリを追加します。インストール後、必要ならカーネルを再起動します。


In [1]:
%pip install "mujoco-warp==3.14.0" pandas pyyaml wandb tensorboard


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 61.5 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: click━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [googleapis-common-protos]
    Found existing installation: click 8.1.7━━━━━━━━━━━━━━━━━━  3/11 [googleapis-common-protos]
    Uninstalling click-8.1.7:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [googleapis-common-protos]
      Successfully uninstalled click-8.1.7━━━━━━━━━━━━━━━━━━━━  3/11 [googleapis-common-protos]
  Attempting uninstall: mujoco-warp0m╺━━━━━━━━━━━━━━  7/11 [opentelemetry-sdk]onventions]
    Found existing installation: mujoco-warp 3.13.0━━━━━━━━━━━  7/11 [opentelemetry-sdk]
    Uninstalling mujoco-warp-3.13.0:0m╺━━━━━━━━━━━━━━  7/11 [opentelemetry-sdk]
      Successfully uninstalled mujoco-warp-3.13.0━━━━━━━━━━━━━  7/11 [opentelemetry-sdk]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
assert torch.cuda.is_available(), "CUDA 対応の PyTorch と NVIDIA GPU が必要です"
print("PyTorch", torch.__version__, "GPU", torch.cuda.get_device_name(0))


PyTorch 2.5.1+cu121 GPU NVIDIA RTX A6000


## 2. 元 APEX を取得

コード、模倣 CSV、Go2 URDF、Multi-Critic PPO は公開リポジトリの指定コミットを使います。


In [3]:
from pathlib import Path
import subprocess, sys

PACKAGE_DIR = Path.cwd().resolve()
APEX_ROOT = PACKAGE_DIR / "APEX"
APEX_COMMIT = "f35c54a4fec00c03751e0d28187278313cde51d9"
if not APEX_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/marmotlab/APEX.git", str(APEX_ROOT)], check=True)
subprocess.run(["git", "-C", str(APEX_ROOT), "checkout", APEX_COMMIT], check=True)
sys.path.insert(0, str(PACKAGE_DIR))
sys.path.insert(0, str(APEX_ROOT / "rsl_rl"))
print("APEX:", APEX_ROOT)


Cloning into '/workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/apex_mjwarp/APEX'...
Updating files: 100% (412/412), done.


APEX: /workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/apex_mjwarp/APEX


Note: switching to 'f35c54a4fec00c03751e0d28187278313cde51d9'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at f35c54a Update README


## 3. モデルと環境のスモークテスト

初回の MuJoCo Warp コンパイルには時間がかかります。観測は `(N,45)`、critic 観測は `(N,77)`、報酬は `(N,2)` のはずです。


In [4]:
from go2_mjcf import build_go2_mjcf
import mujoco

urdf = APEX_ROOT / "resources/robots/go2/urdf/go2.urdf"
cpu_model = mujoco.MjModel.from_xml_string(build_go2_mjcf(urdf))
print("nq, nv, nu:", cpu_model.nq, cpu_model.nv, cpu_model.nu)
assert (cpu_model.nq, cpu_model.nv, cpu_model.nu) == (19, 18, 12)


nq, nv, nu: 19 18 12


In [5]:
from env import ApexGo2Warp

env = ApexGo2Warp(APEX_ROOT, num_envs=16)
obs, critic_obs = env.reset()
for _ in range(5):
    obs, critic_obs, reward, done, info = env.step(torch.zeros(16, 12, device="cuda"))
assert obs.shape == (16, 45)
assert critic_obs.shape == (16, 77)
assert reward.shape == (16, 2)
assert torch.isfinite(obs).all() and torch.isfinite(critic_obs).all() and torch.isfinite(reward).all()
print("OK", obs.shape, critic_obs.shape, reward.shape, "mean reward", reward.mean(dim=0).tolist())


Warp 1.17.0 initialized:
   CUDA Toolkit 12.9, Driver 12.3
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX A6000" (48 GiB, sm_86, mempool enabled)
     "cuda:1"   : "NVIDIA RTX A6000" (48 GiB, sm_86, mempool enabled)
     "cuda:2"   : "NVIDIA RTX A6000" (48 GiB, sm_86, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /root/.cache/warp/1.17.0


/usr/local/lib/python3.10/dist-packages/mujoco_warp/_src/io.py:228: UserWarning: 
        CUDA version < 12.4 detected
        - graph capture may be unreliable for < 12.3
        - conditional graph nodes are not available for < 12.4
          Model.opt.graph_conditional should be set to False
        
  warp_util.check_toolkit_driver()


Module mujoco_warp._src.smooth 8d0d9ab load on device 'cuda:0' took 8840.19 ms  (compiled)
Module _nxn_broadphase__locals__kernel_04e59282 04e5928 load on device 'cuda:0' took 0.65 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_1a0e5f50 1a0e5f5 load on device 'cuda:0' took 24583.66 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_701b0e5b 701b0e5 load on device 'cuda:0' took 4990.58 ms  (compiled)
Module mujoco_warp._src.constraint 8d4d513 load on device 'cuda:0' took 1.25 ms  (cached)
Module _friction_dof__locals__kernel_f5f6cb7e f5f6cb7 load on device 'cuda:0' took 0.87 ms  (cached)
Module _limit_slide_hinge__locals__kernel_3fb01e2e 3fb01e2 load on device 'cuda:0' took 0.84 ms  (cached)
Module _efc_contact_init__locals__kernel_3a7c064e 3a7c064 load on device 'cuda:0' took 197.84 ms  (compiled)
Module _efc_contact_jac_dense__locals__kernel_b0c1fdf8 5fcbd59 load on device 'cuda:0' took 0.77 ms  (cached)
Module _efc_contact_update__locals__kernel_1

## 4. 元の Multi-Critic PPO で 1 回だけ更新

学習器は APEX 内の `rsl_rl` を変更せず利用します。W&B は既定でオフラインです。初回は 64 環境・1 更新で接続を確認します。


In [6]:
from train import make_runner
env, runner = make_runner(APEX_ROOT, num_envs=64, log_dir=PACKAGE_DIR / "logs" / "smoke")
runner.learn(num_learning_iterations=1, init_at_random_ep_len=False)
print("checkpoint:", list((PACKAGE_DIR / "logs" / "smoke").glob("model_*.pt")))


/usr/local/lib/python3.10/dist-packages/mujoco_warp/_src/io.py:228: UserWarning: 
        CUDA version < 12.4 detected
        - graph capture may be unreliable for < 12.3
        - conditional graph nodes are not available for < 12.4
          Model.opt.graph_conditional should be set to False
        
  warp_util.check_toolkit_driver()


################################################################################
                         Learning iteration 0/1                         

                       Computation: 477 steps/s (collection: 2.882s, learning 0.338s)
               Value function loss: 0.0933
                    Surrogate loss: 0.0658
             Mean action noise std: 1.00
--------------------------------------------------------------------------------
                   Total timesteps: 1536
                    Iteration time: 3.22s
                        Total time: 3.22s
                               ETA: 3.2s

checkpoint: [PosixPath('/workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/apex_mjwarp/logs/smoke/model_0.pt'), PosixPath('/workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/apex_mjwarp/logs/smoke/model_1.pt')]


## 5. 本学習

1 更新が通った後で環境数と更新回数を増やします。元設定は 4096 環境・1200 更新ですが、必要な GPU メモリと実行時間は実機で測ってください。以下は手動で実行するセルです。


In [8]:
 env, runner = make_runner(APEX_ROOT, num_envs=4096, log_dir=PACKAGE_DIR / "logs" / "full")
 runner.learn(num_learning_iterations=1200, init_at_random_ep_len=False)


/usr/local/lib/python3.10/dist-packages/mujoco_warp/_src/io.py:228: UserWarning: 
        CUDA version < 12.4 detected
        - graph capture may be unreliable for < 12.3
        - conditional graph nodes are not available for < 12.4
          Model.opt.graph_conditional should be set to False
        
  warp_util.check_toolkit_driver()


Loss/learning_rate,▁
Loss/surrogate,▁
Loss/value_function,▁
Perf/collection time,▁
Perf/learning_time,▁
Perf/total_fps,▁
Policy/mean_noise_std,▁
Train/decap_factor_env0,▁
Loss/learning_rate,1e-05
Loss/surrogate,0.06582
Loss/value_function,0.09331


################################################################################
                       Learning iteration 0/1200                        

                       Computation: 19948 steps/s (collection: 4.555s, learning 0.373s)
               Value function loss: 0.0944
                    Surrogate loss: 0.0327
             Mean action noise std: 1.00
--------------------------------------------------------------------------------
                   Total timesteps: 98304
                    Iteration time: 4.93s
                        Total time: 4.93s
                               ETA: 5913.4s

################################################################################
                       Learning iteration 1/1200                        

                       Computation: 21960 steps/s (collection: 4.126s, learning 0.350s)
               Value function loss: 0.0120
                    Surrogate loss: -0.0003
             Mean action noise std: 1.00
-------